Maintenant, nous allons voir les models d'ia pour la prediciton des retards
Nous allons utilisé skilit learn

In [46]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import joblib

df = pd.read_csv("cleaned_dataset.csv", sep=",", on_bad_lines="skip")

# Feature engineering temporel
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year
df["day_of_week"] = df["Date"].dt.dayofweek
df["is_peak_month"] = df["month"].isin([7, 8, 12]).astype(int)
df = df.drop(columns=["Date"])

# Cible
target = "Average delay of all trains at arrival"
y = pd.to_numeric(df[target], errors="coerce")

# Feature retenue
features = [
    "Departure station",
    "Arrival station",
    "Service",
    "month",
    "year",
    "day_of_week",
    "is_peak_month",
]

X = df[features]

# Encoder les catégorielles
X = pd.get_dummies(X, columns=["Service", "Departure station", "Arrival station"])

# Supprimer les lignes incomplètes
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

Notre IA doit donc s'entrainer et tester. Il lui definie donc des paramètres pour savoir sur combien de pourcents du csv il s'entrainera. Ici on lui definie 0.2 correspondant a 20 % du csv

In [47]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


On definie donc le model de l'ia et son entrainement ici un model Linear Regression


In [48]:
# Modèle
model = LinearRegression()
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 2.40
MSE  : 14.39
R²   : 0.2208


On obtient donc 3 resultats du model 

MAE : mean absolute error il se trompe donc de 1,64 min en moyenne
MSE : mean squared error, la plus haute erreur dans les pires cas
R² : de 0 a 1, determine si le model est pertinent au plus proche de 1, ici il l'est moyennement


On test d'autres modeles

In [49]:
# Modèle
model = MLPRegressor()
model.fit(X_train, y_train)

# Evaluation

y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 2.55
MSE  : 15.03
R²   : 0.1863


In [50]:
model = DecisionTreeRegressor()
model.fit(X_train, y_train)

# Evaluation

y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 2.65
MSE  : 24.16
R²   : -0.3082


In [51]:
model = GradientBoostingRegressor()
model.fit(X_train, y_train)

# Evaluation

y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")


MAE : 2.38
MSE  : 13.84
R²   : 0.2505


In [52]:
# Modèle
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 2.03
MSE  : 12.39
R²   : 0.3293


On choisit donc le model qui a le + de performance qui est le random forest

In [53]:
joblib.dump({"model": model, "columns": X.columns.tolist()}, "model.pkl")
print("Modèle sauvegardé !")



Modèle sauvegardé !


Et faison un test en donnant au modele des valeur et voyons le resultat prédit

In [54]:
# Charger le tout en un seul coup
saved = joblib.load("model.pkl")
model = saved["model"]
model_columns = saved["columns"]

# Remplissage
data = {
    "month": 5,
    "year": 2026,
    "day_of_week": 1,
    "is_peak_month": 1,
    "Service_National": 1,
    "Departure station_Bordeaux Saint Jean": 1,
    "Arrival station_Paris Montparnasse": 1,
}

X_input = pd.DataFrame([data], columns=model_columns).fillna(0)

prediction = model.predict(X_input)
print(f"Retard estimé : {prediction[0]:.1f} minutes.")

Retard estimé : 4.5 minutes.


Nous avons des modeles via sklearn. Maintenant nous allons utilisés xgboost qui permet de creer nous meme des ia avec des variables spécifiques.

In [55]:
import xgboost as xgb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

df = pd.read_csv("cleaned_dataset.csv", sep=",", on_bad_lines="skip")

# Feature engineering temporel
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year
df["day_of_week"] = df["Date"].dt.dayofweek
df["is_peak_month"] = df["month"].isin([7, 8, 12]).astype(int)
df = df.drop(columns=["Date"])

# Cible
target = "Average delay of all trains at arrival"
y = pd.to_numeric(df[target], errors="coerce")

# Feature retenue
features = [
    "Departure station",
    "Arrival station",
    "Service",
    "month",
    "year",
    "day_of_week",
    "is_peak_month",
]

X = df[features]

# Encoder les catégorielles
X = pd.get_dummies(X, columns=["Service", "Departure station", "Arrival station"])

# Supprimer les lignes incomplètes
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle
model = xgb.XGBRegressor(
    n_estimators=10000,
    max_depth=8,
    learning_rate=0.001,
    random_state=42
)

model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"MAE  : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

# Sauvegarde en un seul fichier
joblib.dump({"model": model, "columns": X.columns.tolist()}, "model_entrainé.pkl")
print("Modèle sauvegardé !")

RMSE : 3.54
MAE  : 2.12
R²   : 0.3232
Modèle sauvegardé !


Nous allons donc utilisé le model IA pour predire les retard moyens sur une ligne spécifique

In [56]:
import joblib
import pandas as pd

# Charger le tout en un seul coup
saved = joblib.load("model_entrainé.pkl")
model = saved["model"]
model_columns = saved["columns"]

# Remplissage
data = {
    "month": 12,
    "year": 2028,
    "day_of_week": 0,
    "is_peak_month": 0,
    "Service_National": 1,
    "Departure station_Bordeaux Saint Jean": 1,
    "Arrival station_Paris Montparnasse": 1,
}

X_input = pd.DataFrame([data], columns=model_columns).fillna(0)

prediction = model.predict(X_input)
print(f"Retard prédit : {prediction[0]:.1f} minutes")

Retard prédit : 7.7 minutes


Maintenant nous pouvons regarder pour esseyer d'autre prediction comme le pourcentage de chance qu'un train soit annulé

In [57]:
import xgboost as xgb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

df = pd.read_csv("cleaned_dataset.csv", sep=",", on_bad_lines="skip")

# Feature engineering temporel
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year
df["day_of_week"] = df["Date"].dt.dayofweek
df["is_peak_month"] = df["month"].isin([7, 8, 12]).astype(int)
df = df.drop(columns=["Date"])

# Cible
target = "Number of cancelled trains"
y = pd.to_numeric(df[target], errors="coerce")

# Feature retenue
features = [
    "Departure station",
    "Arrival station",
    "Service",
    "month",
    "year",
    "day_of_week",
    "is_peak_month",
]

X = df[features]

# Encoder les catégorielles
X = pd.get_dummies(X, columns=["Service", "Departure station", "Arrival station"])

# Supprimer les lignes incomplètes
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle
model = xgb.XGBRegressor(
    n_estimators=10000,
    max_depth=6,
    learning_rate=0.001,
    random_state=42
)

model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"MAE  : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

# Sauvegarde en un seul fichier
joblib.dump({"model": model, "columns": X.columns.tolist()}, "model_train_cancel.pkl")
print("Modèle sauvegardé !")

RMSE : 13.80
MAE  : 5.52
R²   : 0.6328
Modèle sauvegardé !


In [58]:
import joblib
import pandas as pd

# Charger le tout en un seul coup
saved = joblib.load("model_train_cancel.pkl")
model = saved["model"]
model_columns = saved["columns"]

# Remplissage
data = {
    "month": 12,
    "year": 2028,
    "day_of_week": 0,
    "is_peak_month": 0,
    "Service_National": 1,
    "Departure station_Bordeaux Saint Jean": 1,
    "Arrival station_Paris Montparnasse": 1,
}

X_input = pd.DataFrame([data], columns=model_columns).fillna(0)

prediction = model.predict(X_input)
print(f"Pourcantage de chance d'un train annulé : {prediction[0]:.1f} %")

Pourcantage de chance d'un train annulé : 11.0 %
